# Session 10 — Testing Tool-Using AI Agents

**Halvard Works maintenance assistant.** Four agents — a planner that decides who does what,
then diagnostics, documentation and maintenance — answering one engineer's request each time.

A **tool** is something an agent can call to look something up or do something. A **tool call**
is one use of it, and it has exactly three parts: a **name**, some **arguments**, and a
**return** — whatever comes back.

Sessions before this one looked at *which agent ran*. This one goes one level down and looks
*inside* each agent, at the tools it reached for and what it pointed them at.

> **The sentence this session adds:** *A tool that answers every mistake is a tool that hides
> every mistake.*

Halvard Works is fictional — the plant, its machines, sensors, manuals, parts and history were
invented for this course. Inspired by publicly described industrial-copilot products; not
affiliated with or endorsed by any vendor.

## Every term, defined before it is used

| term | in this session it means |
|---|---|
| **tool** | something an agent can call. This plant has five. |
| **tool call** | one use of a tool: a name, some arguments, and what comes back. |
| **argument** | what a tool is pointed *at*. `machine_id='CONVEYOR'` is an argument. |
| **return** | what the tool hands back. In this plant it is always a string, never an exception. |
| **grant** | the list of tools one agent is allowed to call. Diagnostics cannot search the manual. |
| **run** | one complete execution of one question, start to finish. Has its own tool calls, time, cost and answer. |
| **repeat** | running the same question again as a second, separate run. Three repeats means three runs, not one run measured three times. |
| **self-healing** | a return that reports a mistake *and* hands back what you should have asked for. |
| **hallucinated tool** | a tool name that does not exist. It does not crash; see below. |
| **answer key** | a hand-written record of what a correct run would have looked up. |
| **`must_call`** | part of an answer key: tools an agent has to call at least once. |
| **`must_cover`** | part of an answer key: values that must be passed to *some* tool. |
| **match mode** | how strictly the key is compared. `subset` = at least these. `exact` = these and no others. |
| **evaluator** | a check that reads a run and returns pass, fail, or not-applicable. |
| **arm** | a set of runs with one specific thing deliberately broken in them. |
| **seed** | the code that breaks it. Deterministic, so the same arm is the same every time. |
| **control** | the arm with nothing broken. Everything else is compared against it. |
| **separation** | how much more often a check fires on a broken arm than on the control. |
| **wobble** | how often a check disagrees with itself on the same unchanged run. |
| **judge** | an evaluator that is a language model reading the run and giving a verdict. |
| **reference-free** | a judge that is never shown the answer key. Both judges here are. |

Nothing below uses a word that is not in this table.

## How this session runs

Five short stretches with your hands on the keyboard, not one long one at the end.
Every one of them costs nothing to run and works whether your key is Anthropic, OpenAI or
Gemini.

| | you do | you will know you got it when |
|---|---|---|
| **1** | predict how many tool calls a question took | the three repeats disagree with each other |
| **2** | break three tool calls on purpose | not one of them raises |
| **3** | find which check fails on a run that looks fine | two checks disagree about the same run |
| **4** | write an answer key for two questions | the screener calls it DISCRIMINATING |
| **5** | disagree with a real judge verdict | you find out what the judge said about a clean run |

Write things on paper before you run the cell that answers them. The gap between your guess
and the output is the whole point, and you only get it once per question.

## Setting up

This cell puts the course's shared folders on the import path and loads the modules this
session uses. It calls no model and costs nothing.

**What to look for:** a version line and a row count. If the row count is not 12, stop and
say so — everything after this depends on those twelve questions being loaded.

In [ ]:
# [SETUP]
import _path  # noqa: F401  -- must be first; puts shared/ and plant/ on the path

import json

import seeds10
import stub_tools10 as st
import tool_eval10 as te
import tool_rows10 as tr

RUNS = json.loads((_path.session(7) / 'runs7.json').read_text())['runs']

print('tool_rows10 ', tr.__version__, '  rows:', len(tr.ROWS))
print('tool_eval10 ', te.__version__, '  checks:', len(te.OFFLINE))
print('seeds10     ', seeds10.__version__, '  arms:', len(seeds10.ARMS))
print('runs loaded ', len(RUNS))

## A tool call has three parts

Before anything else, look at one. The cell below calls a real tool with a real argument and
prints what comes back.

**What to look for:** the three parts, separately. `sensor_history` is the **name**.
`machine_id='CONVEYOR'` is the **argument**. Everything printed after that is the **return**.

Notice that the return is a *string* — a piece of text the agent then has to read. It is not a
Python object and it is not an error code. That matters more than it looks like it does, and
you will see why in a few cells.

In [ ]:
# [ONE_CALL]
from plant_tools7 import sensor_history

print(sensor_history.invoke({'machine_id': 'CONVEYOR'})[:600])

## The five tools, and who is allowed to call each one

Every agent is given a fixed list of tools. It cannot call anything outside that list — or
rather, it can *try*, and you will see in a moment what happens when it does.

**What to look for:** diagnostics cannot search the manual, and documentation cannot read a
sensor. Somebody decided that when this system was designed. It is a choice, not a law, and
it is the kind of choice this session is about testing.

In [ ]:
# [TOOLS]
for agent in ('diagnostics', 'documentation', 'maintenance'):
    print(f'{agent:<14} {sorted(st.GRANTS[agent])}')

print()
print('all five tools:', sorted(st.TOOL_NAMES))

---

# Beat 1 — predict the count

**Three minutes. Paper first.**

Two of the twelve questions:

- **HW-003** — *"The filler drive keeps tripping on overcurrent at shift start. Can it wait
  until the planned stop?"*
- **HW-006** — *"Doing the morning walkdown — check the filler infeed conveyor and the
  air-knife blower and tell me which needs attention first."*

Each of these was run **three separate times**, unchanged — same question, same agents, same
settings. Three complete runs, start to finish. Not one run measured three ways.

**Write down, on paper, how many tool calls you think each one made.** One number each.

Do not run the next cell until both numbers are written down.

In [ ]:
# [PREDICT]
# Each of the three is a SEPARATE, COMPLETE run of the same question -- its own
# tool calls, its own time, its own cost, its own answer. Not an average of
# anything, and not a running total.
for rid in ('HW-003', 'HW-006'):
    runs = {r['rep']: (r['outputs'].get('metrics') or {})
            for r in RUNS if r.get('phase') == 'comparison'
            and r['row_id'] == rid and r['arm'] == 'pipeline'}
    print(rid)
    for n in (1, 2, 3):
        m = runs.get(n, {})
        # A missing value prints as 'not recorded', NEVER as 0. One of these six
        # runs genuinely has no cost on it, and a fabricated $0.0000 would be the
        # most confident wrong number on the screen.
        cost = m.get('cost_usd')
        cost_txt = f'${cost:.4f}' if cost is not None else 'cost not recorded'
        lat = m.get('latency_s')
        lat_txt = f'{lat:.1f}s' if lat is not None else '  --  '
        print(f'    run {n} of 3:  {m.get("n_tool_calls"):>2} tool calls   '
              f'{lat_txt:>7}   {cost_txt}')
    print()

### One minute with your partner, on the second question

1. How far off was your number?
2. **Which of the three runs is the right one to write down as the correct answer?**

There is no good answer to the second one. Carry it with you — it decides what the answer key
you write later has to look like.

## And it is not just those two

The cell below does the same count for every question and both versions of the system.

There are **12 questions** and **2 versions of the system** — the four-agent pipeline and a
single agent doing the same job. 12 × 2 = **24 combinations**, and each combination was run
three times. That is 72 runs in total, and all 72 are already on disk.

**What to look for:** for each of those 24 combinations, did its three runs agree? The cell
answers that twice — once for the order the agents ran in, once for how many tool calls
happened. One of those two numbers is zero and the other is not.

In [ ]:
# [REPEATS]
from collections import defaultdict

COMP = [r for r in RUNS if r.get('phase') == 'comparison']
tools_by, paths_by = defaultdict(dict), defaultdict(dict)
for r in COMP:
    key = (r['arm'], r['row_id'])
    tools_by[key][r['rep']] = (r['outputs'].get('metrics') or {}).get('n_tool_calls')
    paths_by[key][r['rep']] = tuple(r['outputs'].get('agent_calls') or [])

print(f"{'arm':<10}{'question':<10}{'tool calls, 3 repeats':<26}{'agent order changed?'}")
for key in sorted(tools_by):
    reps = [tools_by[key].get(i) for i in (1, 2, 3)]
    same_path = len(set(paths_by[key].values())) == 1
    mark = '' if len(set(reps)) == 1 else '  <-- moved'
    print(f'{key[0]:<10}{key[1]:<10}{str(reps):<26}{"no" if same_path else "YES"}{mark}')

tool_varies = sum(1 for v in tools_by.values() if len(set(v.values())) > 1)
path_varies = sum(1 for v in paths_by.values() if len(set(v.values())) > 1)

# Spell out where the denominator comes from. A bare 'of 24' makes the reader
# stop and work it out, and most of them will not.
questions = sorted({r['row_id'] for r in COMP})
versions = sorted({r['arm'] for r in COMP})
print()
print(f'{len(questions)} questions x {len(versions)} versions of the system '
      f'({" and ".join(versions)}) = {len(tools_by)} combinations,')
print(f'each run 3 times = {len(COMP)} runs in total.')
print()
print(f'Of those {len(tools_by)} combinations, how many had all three runs agree?')
print(f'  agent order      the same in {len(paths_by) - path_varies}, '
      f'different in {path_varies}')
print(f'  tool-call count  the same in {len(tools_by) - tool_varies}, '
      f'different in {tool_varies}')

### Why that decides the shape of everything after it

The agents run in the same order every time. The tool calls underneath them do not.

So when you write down *what a correct run should have done*, you cannot write a list in order —
there is no single order to write. You have to write a **set** of tools, plus a **match mode**:
a rule for how strictly to compare it.

That is the whole reason the answer key in this session looks different from a list.

---

# Beat 2 — break three tool calls on purpose

**Five minutes. Your hands, not mine.**

The cell below calls three tools with arguments that are wrong on purpose: a machine that does
not exist, a part that is not stocked, and a manual search for a word that is not in the
manual.

**Before you run it, say out loud to your partner what you expect to happen.** An exception? An
empty result? An error code?

Then run it, and read the *second line* of each return.

In [ ]:
# [BREAK_IT]
from plant_tools7 import equipment_kb, manual_search, parts_inventory

# CHANGE THESE. Try to find an argument that makes one of them raise an exception.
BAD_MACHINE = 'LINE4-GEARBOX'
BAD_PART = 'NONE'
BAD_QUERY = 'imbalance'

for label, out in [
    (f'equipment_kb({BAD_MACHINE!r})',
     equipment_kb.invoke({'machine_id': BAD_MACHINE})),
    (f'parts_inventory({BAD_PART!r})',
     parts_inventory.invoke({'part_no': BAD_PART})),
    (f'manual_search({BAD_QUERY!r}, BLOWER)',
     manual_search.invoke({'query': BAD_QUERY, 'machine_id': 'BLOWER'})),
]:
    print(f'--- {label} ---')
    print(out)
    print()

### Spend two minutes trying to break one of them

Change `BAD_MACHINE`, `BAD_PART` and `BAD_QUERY` to anything you like and run it again. Empty
string. A number. A sentence. Something rude.

**See if you can make any of the three raise an exception.**

When your partner thinks they have one, check it together — an exception is a traceback, not
a reply that contains the word 'error'.

### What you should have found

Nothing raises. Whatever you passed, the tool replied — and the reply told you the mistake
*and handed back what you should have asked for instead*: the list of real machines, the list
of stocked parts, or the note 'no match; try fewer or broader words'.

That is what **self-healing** means in this session: an agent reading that reply can simply
call again and get it right. The mistake costs one extra tool call and then disappears — it
never appears in the final answer, and nothing is logged as a failure, because nothing failed.

Hold on to that. It is the reason this whole session exists.

## And a tool name that does not exist at all

You might expect a made-up tool name to crash the agent. It does not.

The cell below builds the piece of machinery that actually runs tool calls, and sends it two
calls. The first names `vibration_api`, a tool nobody was ever given. The second names a real
tool but points it at a machine that does not exist.

**What to look for: the `status` line on each one.** They are not the same, and the difference
is the wrong way round from what you would want.

*(A note on the code: the tool-running node cannot be called on its own — it expects to be
inside a graph — so the cell wraps it in the smallest graph that will hold it. That is why
there are four lines of scaffolding before anything interesting happens.)*

In [ ]:
# [HALLUCINATED]
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

from plant_tools7 import TOOLS_DIAGNOSTICS

# ToolNode expects to run inside a graph, so give it the smallest one that works.
g = StateGraph(MessagesState)
g.add_node('tools', ToolNode(TOOLS_DIAGNOSTICS))
g.add_edge(START, 'tools')
g.add_edge('tools', END)
runner = g.compile()

def send(name, args):
    msg = AIMessage(content='', tool_calls=[
        {'name': name, 'args': args, 'id': 'call_1'}])
    reply = runner.invoke({'messages': [msg]})['messages'][-1]
    print(f'called  : {name}({args})')
    print(f'status  : {reply.status}')
    print(f'content : {reply.content[:220]}')
    print()

send('vibration_api', {'machine_id': 'CONVEYOR'})   # a tool that does not exist
send('equipment_kb', {'machine_id': 'LINE4-GEARBOX'})  # a real tool, bad argument

### Read the two `status` lines again

| what was wrong | `status` |
|---|---|
| the tool name does not exist | `error` |
| the tool is real, the machine is not | `success` |

A made-up tool name gets flagged. **A real tool pointed at something that does not exist is
reported as a success**, with the problem written inside the text of the reply where no status
field will ever see it.

So the one signal that *is* countable — the status flag — fires for the mistake an agent is
least likely to make, and stays quiet for the one it makes all the time.

## So somebody has to write down what the question needed

If the mistakes do not reach the answer, the only place they are still visible is the list of
tool calls — and reading that list only tells you something if you know what *should* have been
in it.

That record is an **answer key**. Twelve of them were written by hand for this plant, one per
question. The cell below prints one.

**What to look for:** three fields. `must_call` is which tools, per agent. `must_cover` is which
*values* have to be passed to some tool. `match_mode` is how strictly to compare.

And read the `why`. Every key was written from the **question**, not from watching a run. A key
copied from a transcript can only ever agree with itself.

In [ ]:
# [ROW]
row = tr.BY_ID['HW-001']

print('question :', [r['question'] for r in RUNS if r['row_id'] == 'HW-001'][0])
print()
for agent, tools in row['must_call'].items():
    print(f'  must call    {agent:<14} {sorted(tools)}')
print(f'  must cover   {row["must_cover"]}')
print(f'  match mode   {row["match_mode"]}')
print()
print('why this row looks like this:')
print(' ', row['why'])

## Four checks, and not one of them calls a model

Given a key and a run, four ordinary Python functions can say whether the run did what the
question needed. Each returns `1` for pass, `0` for fail, or `None` when the key says nothing
about that question.

| check | the question it asks |
|---|---|
| `tool_selection` | which tools were called |
| `tool_arguments` | what they were called with |
| `tool_self_heal` | how many calls came back empty or as an error |
| `tool_result_used` | did what came back reach the answer |

The split between the first two is the one to hold on to. The next cell is where it earns its
keep.

In [ ]:
# [EVAL]
def show(row_id, calls=None):
    rec = next(r for r in RUNS if r.get('phase') == 'matrix'
               and r.get('seed') == 'healthy' and r.get('row_id') == row_id)
    out = st.with_tools(rec['outputs'])
    if calls is not None:
        out = dict(out, tool_calls=calls)
    print(f'{row_id}: {rec["question"]}')
    print()
    # The plan is printed too. Without it the tool calls look fine, and you have
    # to hold the question in your head to spot what is missing.
    print('  what the planner decided to do:')
    for i, step in enumerate(out.get('plan') or [], 1):
        print(f'    {i}. {step["agent"]:<14} {step["subtask"]}')
    print()
    print('  what was actually looked up:')
    print(st.render(out['tool_calls']))
    print()
    for k, v in te.run_all(out, tr.BY_ID[row_id]).items():
        mark = {1: 'pass', 0: 'FAIL', None: ' -- '}[v['score']]
        print(f'  {mark}  {k:<18} {v["comment"]}')
    return out

_ = show('HW-006')

### Read that output before going on

`tool_selection` **passed**. Every tool the question needed was called.

`tool_arguments` **failed**. The question names two machines. Diagnostics ran twice — which is
correct, there are two machines — but both times it looked up the same one. The other machine
is never passed to any tool in the entire run.

Counting the calls says two. Reading the arguments says both calls were for the same thing.
That is the difference between the two checks, on one run, at the same moment.

The answer this run produced still reads perfectly well.

---

# Beat 3 — find the check that fails

**Six minutes. Two people, one screen.**

The cell below runs all four checks on any question you name. Run it on these three, one at a
time, and for each one **write down which check fails before you read the comment**:

`HW-005`  ·  `HW-011`  ·  `HW-012`

All three produced an answer that reads perfectly well. All three pass every check this course
had before today.

**The question to answer for each one:** is the check being fussy, or did the run genuinely
miss something the engineer asked for? Argue about it with your partner before moving on.

In [ ]:
# [FIND_FAIL]
# Change this and re-run. Try HW-005, then HW-011, then HW-012.
LOOK_AT = 'HW-005'

_ = show(LOOK_AT)

### Take three answers, not one

- **HW-005** — the manual search came back with nothing. The section it wanted is called
  'Balance criteria'; the query was 'imbalance'. Is an empty search a failure, or just a
  wasted call?
- **HW-011** — the engineer asked whether this is the same failure as last September. One tool
  returns past work orders. It was never called. Whose fault is that — the agent's, or the
  person who wrote the key?
- **HW-012** — same empty search as HW-005, different machine.

There is a defensible answer on both sides for at least one of these. That disagreement is
what you are about to have to settle in writing.

## Now score all twelve

The cell below runs all four checks over every question, on the runs where **nothing was
deliberately broken**. This is the control: the baseline that everything else gets compared to.

**What to look for:** the totals are not 12/12.

In [ ]:
# [CONTROL]
rows_by_id = {r['id']: r for r in tr.ROWS}
healthy = {}
for rec in RUNS:
    if rec.get('phase') == 'matrix' and rec.get('seed') == 'healthy':
        if rec['row_id'] in rows_by_id:
            o = st.with_tools(rec['outputs'])
            o['question'] = rec['question']
            healthy[rec['row_id']] = o

totals = {k: [0, 0] for k in te.KEYS}
fails = []
for rid, out in sorted(healthy.items()):
    res = te.run_all(out, rows_by_id[rid])
    for k in te.KEYS:
        if res[k]['score'] is not None:
            totals[k][1] += 1
            totals[k][0] += res[k]['score']
            if res[k]['score'] == 0:
                fails.append((rid, k, res[k]['comment']))

for k, (ok, n) in totals.items():
    print(f'{k:<20} {ok}/{n}')

print()
print(f'{len(fails)} failure(s) on runs where nothing was deliberately broken:')
for rid, k, comment in fails:
    print(f'  {rid}  {k}')
    print(f'      {comment}')

### What that means

The control was never clean. Several of those twelve questions were answered by a run that
looked up the wrong thing, or never looked something up at all, or ran a search that came back
empty — and every one of them passes every check that existed before today.

Nobody had looked at this layer, so nobody had found them.

---

# Beat 4 — write the answer key

**Fifteen minutes. The main one.**

Open **`session-10/my_tools10.py`**. It is the only file you edit today.

There is a worked example at the top, already filled in, with the reasoning written out clause
by clause. Read it first. Then fill in the two rows below it for HW-003 and HW-006.

Three fields to fill in per row:

- `must_call` — which tools each agent has to call at least once
- `must_cover` — which values have to be passed to some tool
- `match_mode` — `subset` (at least these) or `exact` (these and no others)

**Write it from the question, not from what you just saw printed.** If you write down what the
run did, your key will agree with that run no matter how wrong the run was.

## Step 3: screen it

Run the cell below, or the same command in a terminal. It calls no model and costs nothing, so
run it as often as you like.

It does not ask whether your key matches ours — there is no single right key. It asks the only
question that matters about any check: **does it tell a broken run from a healthy one?**

You get one of three verdicts per row:

- **DISCRIMINATING** — fires on at least one broken run, quiet on the healthy one. This is the
  one you want.
- **DECORATIVE** — never fires on anything. Not wrong, just not testing anything. Usually it
  asks for less than the question needs.
- **WRONG** — fires on the healthy run. In production that is a check that cries wolf on every
  good run, and people stop reading it inside a week.

**Getting DECORATIVE on the first try is normal.** Read which broken runs it stayed quiet on,
and ask what the question needs that your key did not require.

In [ ]:
# [SCREEN]
# Plain Python rather than a %run magic, so this works the same whether you are in
# VS Code, in Jupyter, or running the notebook from the repo root.
import runpy
import sys

sys.argv = ['screen_my_tools.py']
try:
    runpy.run_path(str(_path.ROOT / 'session-10' / 'screen_my_tools.py'),
                   run_name='__main__')
except SystemExit:
    pass   # the script ends with SystemExit(0); that is not an error here

---

## What is left for a model to do?

Be honest about what the four checks already did. *Was this tool called?* is looking in a list.
*Are the arguments well formed?* is a schema check. *Did the result reach the answer?* is a text
search. None of that needs a model, and a model would be slower and cost money.

Two questions are left over, and both of them need somebody to **read the request**:

1. Was this the right tool to reach for in the first place?
2. Did the agent stop looking before it had enough?

Those are the two **judges** — evaluators that are a language model reading the run. Both are
**reference-free**: they are shown the request, the roster of who may call what, and the tool
calls. They are never shown the answer key, and never shown the final answer. That restriction
is the experiment.

The cell below prints the exact text one judge is sent. If you cannot read the prompt, you
cannot argue with the verdict — and arguing with the verdict is the point.

In [ ]:
# [PROMPT]
import judge10

out = healthy['HW-003']
print(judge10.build_prompt('search_sufficiency', out['question'], out['tool_calls']))

---

# Beat 5 — disagree with the judge

**Eight minutes. Vote before you look.**

The two judges have already been run for real, against twenty runs. Those verdicts are saved
on disk, so this costs nothing.

The cell below shows you four of them **with the run hidden**. You get the question, the tool
calls, and what the judge said. You do **not** get to know whether anything was actually broken
in that run.

**For each one, vote with your partner: was the judge right?** Write down yes or no, four
times, before you run the reveal cell.

In [ ]:
# [VERDICTS_BLIND]
import json

VERDICTS = _path.session(10) / 'preflight10_verdicts.json'
if not VERDICTS.exists():
    print('No saved verdicts on this machine. Your instructor has them; skip to the')
    print('reveal cell and read the table there instead.')
else:
    saved = json.loads(VERDICTS.read_text())['verdicts']
    picks = [('HW-001', 'search_sufficiency'), ('HW-003', 'search_sufficiency'),
             ('HW-007', 'tool_fit'), ('HW-002', 'tool_fit')]
    SHOWN = []
    for n, (rid, judge) in enumerate(picks, 1):
        rec = next(r for r in saved if r['row_id'] == rid and r['judge'] == judge
                   and r['arm'] == 'healthy')
        SHOWN.append(rec)
        out = healthy[rid]
        print(f'===== RUN {n} =====')
        print(f'the engineer asked: {out["question"]}')
        print()
        print(st.render(out['tool_calls']))
        print()
        print(f'the {judge} judge said:')
        print(f'  {rec["comment"]}')
        print()
        print('  was the judge right?  yes / no  ->  ____')
        print()

### Do not run the next cell until you have four answers written down.

In [ ]:
# [VERDICTS_REVEAL]
if VERDICTS.exists():
    print('Every one of those four runs was the CONTROL — the version with nothing')
    print('deliberately broken in it.')
    print()
    saved = json.loads(VERDICTS.read_text())['verdicts']
    for judge in sorted({r['judge'] for r in saved}):
        clean = [r for r in saved if r['judge'] == judge and r['arm'] == 'healthy']
        fired = [r for r in clean if r['score'] == 0]
        print(f'{judge:<20} called {len(fired)} of {len(clean)} clean runs UNSOUND')
    print()
    reasons = [r['comment'] for r in saved
               if r['judge'] == 'search_sufficiency' and r['score'] == 0]
    hist = sum(1 for c in reasons if 'maintenance_history' in c)
    print(f'and {hist} of its {len(reasons)} UNSOUND verdicts blame the same thing:')
    print('  maintenance_history was never called.')

### What just happened, and why it is the point of the session

One judge called **every single clean run** unsound. Most of the time for the same reason: the
run never checked past work orders.

**Is it wrong?** Not obviously. 'You diagnosed a recurring fault and never looked at the
history' is a fair thing for an engineer to say. It is a stricter standard than the answer key
uses — the key only demands a history lookup on the one question that asks about last
September.

So the judge and the key disagree about what *enough* means. Consistently, in one direction,
about one tool.

**And a check that fires on three-quarters of everything cannot tell you anything.** It does
not separate good runs from bad ones, because it calls them all bad. That is not the judge
being stupid; it is the judge answering a slightly different question than the one the key
was written for — and nobody noticed until the clean runs were scored.

Which is why you always run your checker against a run you already know is fine.

## Run the judges for free first

There is a stub version of each judge: a few lines of string matching, no model, no cost. It
exists so the plumbing can be checked without spending anything.

**Say this out loud to yourself: this is string matching.** Fooling it is not fooling a model,
and passing it is not passing a model. It is here to show you the shape of a verdict, nothing
more.

**What to look for:** a verdict word, and the single fact the verdict rests on.

In [ ]:
# [JUDGE_STUB]
for arm in seeds10.ARMS:
    o = dict(healthy['HW-003'])
    o['tool_calls'] = seeds10.apply(arm, healthy['HW-003']['tool_calls'], {})
    res = judge10.run_all(o, stub=True)
    print(f'--- {arm} ---')
    for k, v in res.items():
        print(f'  {k:<20} {v["comment"]}')
    print()

## And now for real, if you have a key set

This is the only cell in the notebook that calls a model. The cost is **computed from the code
that makes the calls**, not typed into a comment — run the first two lines and you will see the
number before anything is spent.

It works on Anthropic, OpenAI or Gemini: the `CHAT` alias picks up whichever provider your key
is for. If you have no key, skip this cell — everything above and below it runs without one.

**What to look for:** whether the judge catches the arm where a *different real machine* was
looked up. Nothing errored there, nothing came back empty, and every argument was well formed.
The only code check that catches it is the one reading a key somebody wrote by hand.

In [ ]:
# [JUDGE_LIVE]
n_arms = len(seeds10.ARMS)
cost = n_arms * judge10.LIVE_CALLS_PER_RUN
print(f'this cell makes {n_arms} arms x {judge10.LIVE_CALLS_PER_RUN} judges '
      f'= {cost} model calls')

RUN_IT = False   # set to True when you are ready to spend

if RUN_IT:
    for arm in seeds10.ARMS:
        o = dict(healthy['HW-003'])
        o['tool_calls'] = seeds10.apply(arm, healthy['HW-003']['tool_calls'], {})
        print(f'--- {arm} ---')
        for k, v in judge10.run_all(o, stub=False).items():
            expected = judge10.EXPECT[arm][k]
            mark = 'as expected' if v['score'] == expected else 'DISAGREES with the key'
            print(f'  {k:<20} {v["comment"]}')
            print(f'  {"":<20} ({mark})')
        print()
else:
    print('RUN_IT is False — nothing was spent. Set it to True to run for real.')

---

## What this notebook cannot tell you

Four things, and they all belong on the same page as the findings:

1. **The tool calls on the seeded arms are reconstructed**, not captured live. They are worked
   out from what each agent reads, which is faithful, but it is not a recording.
2. **The numbers are small.** Four questions per arm, one repeat. Every figure is a range.
3. **The seeded arms have frozen answers.** A broken tool call is injected into a saved run, so
   `tool_result_used` is excluded from those arms in both directions — data from an injected
   call could never appear in an answer written before the injection.
4. **One plant, twelve questions.** Nothing here tells you what happens at twelve thousand.

## The sentence to take away

> *A tool that answers every mistake is a tool that hides every mistake.*

Every tool in this plant replies helpfully when you get it wrong. That is good engineering and
it is why the agents recover. It is also why the only place a mistake is still visible is a
list of tool calls that somebody has to count — and counting only means something if somebody
wrote down what the right count was.